In [13]:
import os
import sys
import tempfile
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp

from torch.nn.parallel import DistributedDataParallel as DDP
import numpy as np

# Modelos

In [14]:
def alex_net_creator():

    transform = transforms.Compose([
        transforms.Resize((227, 227)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def create_model():
            alexnet=torchvision.models.alexnet(weights=None)
            criterion = nn.CrossEntropyLoss()
            #optimizer = optim.SGD(alexnet.parameters(), lr=0.005, momentum=0.9)
            optimizer_params={'momentum':0.9, 'lr':0.005}
            return alexnet, criterion, optimizer_params
    print("Tto")
    return transform,'alexnet',create_model

# Dataset

In [15]:
seed=None

def get_train_loaders_from_ds(ds,splits=1,batch_size=64):
    splitted_data = random_split(ds,[len(ds)//splits] * splits,generator=torch.Generator().manual_seed(seed) if seed else None)
    return [DataLoader(data,batch_size=batch_size,shuffle=True) for data in splitted_data]



# Constantes

In [21]:
WORLD_SIZE=4
MODEL_INFO=alex_net_creator

transform,MODEL_NAME,MODEL_CREATOR= MODEL_INFO()
DS=datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
DATA_SET_NAME= 'cifar10'
EPOCHS=10
BATCH_SIZE=32
BACKEND='gloo'


Tto


# Setup node in ddp

In [22]:
def setup_node(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'

    # We want to be able to train our model on an `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
    # such as CUDA, MPS, MTIA, or XPU.
    #acc = torch.accelerator.current_accelerator()
    #backend = torch.distributed.get_default_backend_for_device(acc)
    # initialize the process group
    dist.init_process_group(BACKEND, rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

In [30]:



def train_model(rank, world_size):

    print(f"Running basic DDP example on rank {rank}.")
    try:
        setup_node(rank, world_size)

        sampler = DistributedSampler(DS, num_replicas=world_size, rank=rank)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)
        _model, criterion, optimizer_params = MODEL_CREATOR()
        _model.to(rank)
        ddp_model = DDP(_model, device_ids=[rank])

        optimizer = optim.SGD(ddp_model.parameters(), **optimizer_params)

        for epoch in range(EPOCHS):
            sampler.set_epoch(epoch)

            for data, target in dataloader:
                data, target = data.to(rank), target.to(rank)

                optimizer.zero_grad()
                outputs = ddp_model(data)
                loss = criterion(outputs, target)
                loss.backward()
                optimizer.step()

        cleanup()
        print(f"Finished running basic DDP example on rank {rank}.")
    except Exception as e:
        print(f"Failed running basic DDP example on rank {rank}: {e}")




# Corremos el model

In [31]:
  mp.spawn(train_model,
             args=(WORLD_SIZE,),
             nprocs=WORLD_SIZE,
             join=True)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=68, pipe_handle=84)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/usr/lib64/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'train_model'
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=68, pipe_handle=79)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/usr/lib64/python3.14/multiprocessing/spawn

ProcessExitedException: process 2 terminated with exit code 1